In [6]:
import numpy as np
import pandas as pd
import geopandas as gpd
from io import StringIO

In [2]:
data_gdf = gpd.read_file("/content/MasterPlan2019PlanningAreaBoundaryNoSea.geojson")

data_gdf.head()

,Name,Description,geometry
0,kml_1,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.93208 1.30555 0, 103.93208 1.3..."
1,kml_2,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.72042 1.32824 0, 103.72003 1.3..."
2,kml_3,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.76408 1.37001 0, 103.76444 1.3..."
3,kml_4,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.82361 1.26018 0, 103.82362 1.2..."
4,kml_5,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.77445 1.39029 0, 103.77499 1.3..."


In [7]:
html_str = data_gdf.loc[0, "Description"]

table = pd.read_html(StringIO(f"<html>{html_str}</html>"))[0]
table.columns = ["Attribute", "Value"]
table = table.dropna().reset_index(drop=True)
print(table)

    Attribute             Value
0  PLN_AREA_N             BEDOK
1  PLN_AREA_C                BD
2      CA_IND                 N
3    REGION_N       EAST REGION
4    REGION_C                ER
5     INC_CRC  5F00E6FF084F3364
6  FMEL_UPD_D    20191223152014


In [11]:
def html_to_dict(html_text):
    t = pd.read_html(StringIO(f"<html>{html_text}</html>"))[0]
    t.columns = ["Attribute", "Value"]
    return dict(zip(t["Attribute"], t["Value"]))

expanded = data_gdf["Description"].apply(html_to_dict).apply(pd.Series)

clean_gdf = pd.concat([data_gdf.drop(columns=["Description"]), expanded], axis=1)

clean_gdf.head()

,Name,geometry,PLN_AREA_N,PLN_AREA_C,CA_IND,REGION_N,REGION_C,INC_CRC,FMEL_UPD_D
0,kml_1,"POLYGON Z ((103.93208 1.30555 0, 103.93208 1.3...",BEDOK,BD,N,EAST REGION,ER,5F00E6FF084F3364,20191223152014
1,kml_2,"POLYGON Z ((103.72042 1.32824 0, 103.72003 1.3...",BOON LAY,BL,N,WEST REGION,WR,C96AED188C00B2FC,20191223152014
2,kml_3,"POLYGON Z ((103.76408 1.37001 0, 103.76444 1.3...",BUKIT BATOK,BK,N,WEST REGION,WR,3BEC4C829160F28A,20191223152014
3,kml_4,"POLYGON Z ((103.82361 1.26018 0, 103.82362 1.2...",BUKIT MERAH,BM,N,CENTRAL REGION,CR,4850795BB0B6A4F7,20191223152014
4,kml_5,"POLYGON Z ((103.77445 1.39029 0, 103.77499 1.3...",BUKIT PANJANG,BP,N,WEST REGION,WR,656F87D23D6DAB02,20191223152014


In [12]:
clean_gdf.to_file("/content/clean_MasterPlan2019PlanningAreaBoundaryNoSea.geojson", driver='GeoJSON')